Owen's Results:

The model is unable to detect thin oil (light blue in the RGB)

Mean AUC: 0.9224

Mean DP: 0.6027

Paper Results: 

Mean AUC: 0.9006

Mean DP: 0.8551

In [ ]:
import os
import gc
import time
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.signal import convolve2d
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import cg, LinearOperator

from sklearn.decomposition import KernelPCA
from sklearn.metrics import (
    roc_auc_score, 
    precision_score, 
    recall_score, 
    accuracy_score, 
    f1_score,
    average_precision_score
)

from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Hyperparams
SEED = 42 
D = 25      # PCA Bands
T = 800     # No. of isolation trees
SVM_TRAIN_FRACTION = 0.01
BETA_ERW = 710.0
GAMMA_ERW = 1e-5

# Prevent OOM Errors
N_KPCA_TRAIN = 5000
KPCA_BATCH_SIZE = 1000
SVM_BATCH_SIZE = 20000

In [5]:
from pathlib import Path

DATA_DIR = Path.cwd().parent.parent / "data" / "hyperspectral_oil_spill"
OUTPUT_DIR = DATA_DIR / "kpca-svm-erw-results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

METRICS_CSV = os.path.join(OUTPUT_DIR, "GM01_GM18_metrics.csv")

In [ ]:
M_NOISE = np.array([
    [ 1, -2,  1], 
    [-2,  4, -2], 
    [ 1, -2,  1]
], dtype=np.float64)

def noisy_band_removal(img):
    H, W, B = img.shape
    sigmas = np.zeros(B, dtype=np.float64)
    
    for b in range(B):
        # Applies Equation 1 and 2
        filtered = convolve2d(img[:, :, b], M_NOISE, mode="valid")
        sigmas[b] = (np.sqrt(np.pi / 2.0) * np.sum(np.abs(filtered)) / (6.0 * (W - 2) * (H - 2)))
        
    # Applies Equation 3
    threshold = np.sum(sigmas) / (2.0 * B)
    keep_mask = sigmas < threshold
    
    selected_bands = np.where(keep_mask)[0]
    img_selected = img[:, :, keep_mask]
    
    return img_selected, selected_bands, threshold

In [ ]:
def run_main_kpca(img_selected):
    H, W, B_selected = img_selected.shape
    X = img_selected.reshape(-1, B_selected)
    N = X.shape[0]
    
    rng = np.random.RandomState(SEED)
    kpca_idx = rng.choice(N, size=min(N_KPCA_TRAIN, N), replace=False)
    X_train = np.asarray(X[kpca_idx], dtype=np.float64)
    
    # Relying on standard sklearn defaults (gamma = 1 / n_features) to align with paper
    kpca = KernelPCA(
        n_components=D, 
        kernel="rbf", 
        eigen_solver="arpack", 
        fit_inverse_transform=False, 
        remove_zero_eig=True, 
        n_jobs=-1
    )
    kpca.fit(X_train)
    
    Y = np.empty((N, D), dtype=np.float32)
    for start in range(0, N, KPCA_BATCH_SIZE):
        end = min(start + KPCA_BATCH_SIZE, N)
        Y[start:end] = kpca.transform(X[start:end]).astype(np.float32)
        
    # We return the default gamma (1 / B_selected) so it doesn't break the logging dictionary in Cell 12 later
    return Y, (1.0 / B_selected)

In [ ]:
def run_iforest_kmeans(Y):
    # Initializes the T=800 trees specified in the paper parameters
    iso = IsolationForest(
        n_estimators=T, 
        max_samples="auto", 
        bootstrap=False, 
        random_state=SEED, 
        n_jobs=-1
    )
    iso.fit(Y)
    
    # Negating score_samples perfectly yields the paper's Equation 6
    probability_map = -iso.score_samples(Y)
    
    # Applies K-Means directly to the probability map as specified in Section 2.2
    kmeans = KMeans(n_clusters=2, n_init=10, random_state=SEED)
    clusters = kmeans.fit_predict(probability_map.reshape(-1, 1))
    centers = kmeans.cluster_centers_.ravel()
    
    # Oil has the higher probability score
    oil_cluster = np.argmax(centers)
    pseudo_labels = (clusters == oil_cluster).astype(np.uint8)
    
    return probability_map, pseudo_labels, centers

In [ ]:
def select_training_set(Y, pseudo_labels):
    N = Y.shape[0]
    
    # Calculates 1% of the total dataset
    N_train = int(np.ceil(SVM_TRAIN_FRACTION * N))
    
    rng = np.random.RandomState(SEED)
    
    # Randomly selects 1% of the samples directly from Tr
    svm_idx = rng.choice(N, size=N_train, replace=False)
    
    X_train = Y[svm_idx]
    y_train = pseudo_labels[svm_idx]
    
    return X_train, y_train, svm_idx

In [ ]:
def run_svm(Y, pseudo_labels):
    X_train, y_train, svm_idx = select_training_set(Y, pseudo_labels)
    
    # Feature scaling (Standard ML practice for RBF-SVMs)
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    Y_scaled = scaler.transform(Y)
    
    # 5-fold CV as specified in Section 2.3
    param_grid = {"C": 10.0 ** np.arange(-2, 5), "gamma": 2.0 ** np.arange(-3, 5)}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    
    # GridSearchCV automatically refits the best model using the full training set
    grid = GridSearchCV(
        SVC(kernel="rbf", probability=True, random_state=SEED), 
        param_grid=param_grid, 
        cv=cv, 
        n_jobs=-1
    )
    grid.fit(X_train_scaled, y_train)
    
    best_svm = grid.best_estimator_
    best_C, best_gamma = grid.best_params_["C"], grid.best_params_["gamma"]
    
    # Pre-allocate output array
    svm_prob = np.empty((Y.shape[0], 2), dtype=np.float32)
    class_to_col = {cls: i for i, cls in enumerate(best_svm.classes_)}
    
    # Batched predictions to prevent RAM overload
    for start in range(0, Y.shape[0], SVM_BATCH_SIZE):
        end = min(start + SVM_BATCH_SIZE, Y.shape[0])
        batch_prob = best_svm.predict_proba(Y_scaled[start:end])
        svm_prob[start:end, 0] = batch_prob[:, class_to_col[0]]
        svm_prob[start:end, 1] = batch_prob[:, class_to_col[1]]
        
    return svm_prob, svm_idx, y_train, best_C, best_gamma, grid.best_score_

In [ ]:
def run_erw(Y, svm_prob, svm_idx, y_train, H, W):
    N = H * W
    pixel_grid = np.arange(N, dtype=np.int32).reshape(H, W)
    
    # Get adjacent pixel indices (horizontal, vertical, and diagonals)
    u = np.concatenate([
        pixel_grid[:, :-1].ravel(), 
        pixel_grid[:-1, :].ravel(), 
        pixel_grid[:-1, :-1].ravel(), 
        pixel_grid[:-1, 1:].ravel()
    ]).astype(np.int32, copy=False)
    
    v = np.concatenate([
        pixel_grid[:, 1:].ravel(), 
        pixel_grid[1:, :].ravel(), 
        pixel_grid[1:, 1:].ravel(), 
        pixel_grid[1:, :-1].ravel()
    ]).astype(np.int32, copy=False)
    del pixel_grid

    # Applies Equation 9 directly using squared distance between pixel features
    dist_sq = np.sum((Y[u] - Y[v])**2, axis=1)
    weights = (np.exp(-BETA_ERW * dist_sq) + 1e-5).astype(np.float64)
    del dist_sq

    # ERW Sparse Matrix Construction
    degree = np.bincount(u, weights=weights, minlength=N) + np.bincount(v, weights=weights, minlength=N)
    seeds, seed_labels = np.asarray(svm_idx, dtype=np.int64), np.asarray(y_train, dtype=np.uint8)
    seed_mask = np.zeros(N, dtype=bool)
    seed_mask[seeds] = True

    unknown_idx = np.flatnonzero(~seed_mask)
    N_unknown = len(unknown_idx)
    unknown_position = np.full(N, -1, dtype=np.int32)
    unknown_position[unknown_idx] = np.arange(N_unknown, dtype=np.int32)

    uu_mask = (~seed_mask[u]) & (~seed_mask[v])
    u_uu, v_uu, w_uu = unknown_position[u[uu_mask]], unknown_position[v[uu_mask]], weights[uu_mask]
    
    # Applies Equation 7 and 10 incorporating the aspatial term (GAMMA_ERW)
    diag_values = degree[unknown_idx] + GAMMA_ERW * (svm_prob[:, 0] + svm_prob[:, 1])[unknown_idx]
    diag_idx = np.arange(N_unknown, dtype=np.int32)

    rows, cols = np.concatenate([u_uu, v_uu, diag_idx]), np.concatenate([v_uu, u_uu, diag_idx])
    values = np.concatenate([-w_uu, -w_uu, diag_values])
    A_erw = coo_matrix((values, (rows, cols)), shape=(N_unknown, N_unknown), dtype=np.float64).tocsr()
    A_erw.sum_duplicates()
    del uu_mask, u_uu, v_uu, w_uu, rows, cols, values, diag_values, diag_idx; gc.collect()

    rhs = (GAMMA_ERW * svm_prob[unknown_idx].astype(np.float64))
    seed_class_image = np.full(N, -1, dtype=np.int8)
    seed_class_image[seeds] = seed_labels

    mask_1 = (~seed_mask[u]) & seed_mask[v]
    np.add.at(rhs, (unknown_position[u[mask_1]], seed_class_image[v[mask_1]]), weights[mask_1])

    mask_2 = (~seed_mask[v]) & seed_mask[u]
    np.add.at(rhs, (unknown_position[v[mask_2]], seed_class_image[u[mask_2]]), weights[mask_2])
    del mask_1, mask_2, seed_class_image, u, v, weights, degree; gc.collect()

    # Preconditioner for faster convergence
    M_inv = 1.0 / A_erw.diagonal()
    preconditioner = LinearOperator(shape=A_erw.shape, matvec=lambda x: M_inv * x, dtype=np.float64)

    P_unknown = np.empty((N_unknown, 2), dtype=np.float64)
    cg_info = []
    
    # Solving the system for both oil and water probabilities
    for cls in [0, 1]:
        solution, info = cg(A_erw, rhs[:, cls], M=preconditioner, rtol=1e-6, atol=0.0, maxiter=2000)
        P_unknown[:, cls] = solution
        cg_info.append(info)

    P_erw = np.zeros((N, 2), dtype=np.float64)
    P_erw[unknown_idx] = P_unknown
    P_erw[seeds[seed_labels == 0], 0], P_erw[seeds[seed_labels == 1], 1] = 1.0, 1.0
    P_erw = np.clip(P_erw, 0.0, None)
    
    # Normalize probabilities
    P_erw = P_erw / np.maximum(P_erw.sum(axis=1, keepdims=True), 1e-12)
    
    return P_erw[:, 1].astype(np.float32), cg_info

In [ ]:
def process_scene(scene_name, path):
    """
    Processes a single hyperspectral image scene and extracts the following metrics for CSV logging:
    
    Ablation Metrics:
    - auc_before / ap_before: Tracks SVM performance before ERW spatial optimization.
    
    Hyperparameter States:
    - best_C / best_gamma: The optimal SVM decision boundary parameters found via GridSearch.
    - cv_accuracy: The confidence of the SVM during training.
    
    Data Integrity & Thresholds:
    - B_selected: Number of bands kept after noisy band removal.
    - noise_threshold: The threshold used to filter out corrupted bands.
    
    Class Imbalance Trackers:
    - pseudo_oil_pct: Percentage of oil pixels clustered by the Isolation Forest.
    - training_oil_pct: Percentage of oil pixels in the 1% SVM training subset.
    
    Computational Benchmarks:
    - elapsed_minutes: Total time taken to process the scene.
    """
    start_time = time.time()
    print(f"\n{'='*65}\nPROCESSING {scene_name}\n{'='*65}")
    
    data = sio.loadmat(path)
    img, gt = np.asarray(data["img"], dtype=np.float64), np.asarray(data["map"]).squeeze().astype(np.uint8)
    H, W, B = img.shape
    if gt.shape != (H, W): raise RuntimeError(f"{scene_name}: GT shape mismatch.")
    print(f"Input: {H} x {W} x {B}")

    print("[1/6] Noisy-band removal")
    img_selected, selected_bands, noise_threshold = noisy_band_removal(img)
    B_selected = img_selected.shape[2]
    print(f"      {B} -> {B_selected} bands")
    del img; gc.collect()

    print("[2/6] Main RBF-KPCA")
    Y, _ = run_main_kpca(img_selected)
    del img_selected; gc.collect()

    print("[3/6] Isolation Forest + K-means")
    _, pseudo_labels, _ = run_iforest_kmeans(Y)
    pseudo_oil_pct = 100.0 * pseudo_labels.mean()

    print("[4/6] 1% RBF-SVM")
    svm_prob, svm_idx, y_train, best_C, best_gamma, cv_accuracy = run_svm(Y, pseudo_labels)
    svm_oil_prob, training_oil_pct, gt_flat = svm_prob[:, 1], 100.0 * y_train.mean(), gt.ravel()
    auc_before, ap_before = roc_auc_score(gt_flat, svm_oil_prob), average_precision_score(gt_flat, svm_oil_prob)

    print("[5/6] Extended Random Walker (ERW)")
    erw_oil_score, _ = run_erw(Y, svm_prob, svm_idx, y_train, H, W)

    print("[6/6] Final metrics")
    final_pred = (erw_oil_score >= 0.5).astype(np.uint8)
    
    auc_after = roc_auc_score(gt_flat, erw_oil_score)
    precision = precision_score(gt_flat, final_pred, zero_division=0)
    recall = recall_score(gt_flat, final_pred, zero_division=0)
    accuracy = accuracy_score(gt_flat, final_pred)
    f1 = f1_score(gt_flat, final_pred, zero_division=0)
    
    predicted_oil_pct = 100.0 * final_pred.mean()
    elapsed_minutes = (time.time() - start_time) / 60.0

    print(f"\nAUC: {auc_after:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
    print(f"Accuracy: {accuracy:.4f} | F1-Score: {f1:.4f}")
    print(f"Predicted oil: {predicted_oil_pct:.3f}% | Runtime: {elapsed_minutes:.1f} min")

    metrics_dict = {
        "B_selected": B_selected,
        "noise_threshold": noise_threshold,
        "pseudo_oil_pct": pseudo_oil_pct,
        "training_oil_pct": training_oil_pct,
        "best_C": best_C,
        "best_gamma": best_gamma,
        "cv_accuracy": cv_accuracy,
        "auc_before": auc_before,
        "ap_before": ap_before,
        "auc_after": auc_after,
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy,
        "f1": f1,
        "predicted_oil_pct": predicted_oil_pct,
        "elapsed_minutes": elapsed_minutes
    }

    return erw_oil_score, final_pred, metrics_dict

In [ ]:
import pandas as pd

# Expanded to run all 18 scenes to reproduce the paper's full results
TARGET_SCENES = list(range(1, 19))
pipeline_results = {}
all_metrics = []

print(f"Starting pipeline for scenes: {TARGET_SCENES}")

for scene_number in TARGET_SCENES:
    scene_name = f"GM{scene_number:02d}"
    path = DATA_DIR / f"{scene_name}.mat"
    
    if not path.exists():
        print(f"Skipping {scene_name} - File not found.")
        continue
        
    try:
        erw_score, final_pred, metrics = process_scene(scene_name, path)
        pipeline_results[scene_name] = final_pred
        
        # Attach the scene name to our metrics and store them
        metrics["scene"] = scene_name
        all_metrics.append(metrics)
        
    except Exception as e:
        print(f"\nERROR processing {scene_name}: {e}")

# Compile all collected metrics and export to the CSV defined in Cell 2
if all_metrics:
    metrics_df = pd.DataFrame(all_metrics)
    # Move the 'scene' column to the front for easier reading
    cols = ['scene'] + [c for c in metrics_df.columns if c != 'scene']
    metrics_df = metrics_df[cols]
    
    metrics_df.to_csv(METRICS_CSV, index=False)
    print(f"\nAll scene metrics successfully saved to: {METRICS_CSV}")

In [ ]:
def get_enhanced_rgb(img_array, rgb_bands=[29, 19, 9]):
    rgb_raw = img_array[:, :, rgb_bands].astype(np.float32)
    rgb_clean = np.where(rgb_raw < 0, 0, rgb_raw)
    
    rgb_enhanced = np.zeros_like(rgb_clean)
    for c in range(3):
        channel = rgb_clean[:, :, c]
        p_low, p_high = np.percentile(channel[channel > 0], (2, 98))
        rgb_enhanced[:, :, c] = np.clip((channel - p_low) / (p_high - p_low + 1e-8), 0, 1)
        
    return rgb_enhanced

def plot_pipeline_predictions(scene_name, prediction_map):
    path = DATA_DIR / f"{scene_name}.mat"
    mat_data = sio.loadmat(path)
    
    img = np.asarray(mat_data["img"], dtype=np.float64)
    gt_map = np.asarray(mat_data["map"]).squeeze().astype(np.uint8)
    
    rgb_enhanced = get_enhanced_rgb(img)
    
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    
    axs[0].imshow(rgb_enhanced)
    axs[0].set_title(f"{scene_name} - Actual Data (RGB Enhanced)")
    axs[0].axis("off")
    
    axs[1].imshow(gt_map, cmap="inferno")
    axs[1].set_title("Ground Truth Mask")
    axs[1].axis("off")
    
    axs[2].imshow(prediction_map, cmap="inferno")
    axs[2].set_title("Unsupervised Pipeline Prediction")
    axs[2].axis("off")
    
    plt.tight_layout()
    plt.show()

for scene, pred in pipeline_results.items():
    plot_pipeline_predictions(scene, pred)

scene: The target identifier (GM01 through GM18).
B_selected: The total number of spectral bands that survived the noisy-band removal.
noise_threshold: The calculated mathematical threshold used to filter those corrupted bands.
pseudo_oil_pct: The percentage of pixels the Isolation Forest and K-means assigned to the "oil" cluster for that entire image.
training_oil_pct: The percentage of positive oil samples in the randomly selected 1% SVM training subset.
best_C: The optimal regularization parameter GridSearch locked in for the RBF-SVM.
best_gamma: The optimal kernel coefficient GridSearch selected for the RBF-SVM.
cv_accuracy: The SVM's internal 5-fold cross-validation score on the training subset.
auc_before: The initial ROC-AUC score based purely on the SVM's probability map.
ap_before: The initial Average Precision score based purely on the SVM's probability map.
auc_after: The final ROC-AUC score after applying the spatial Extended Random Walker (ERW) optimization.
precision: The final positive predictive value (this exactly matches the "Detection Precision" or DP metric from the paper). 
recall: The final sensitivity score tracking how much total oil was successfully captured.
accuracy: The final percentage of total pixels (both oil and water) correctly classified.
f1: The harmonic mean of precision and recall for the final prediction map. 
predicted_oil_pct: The ultimate percentage of pixels the pipeline flagged as oil in the final output map. 
elapsed_minutes: The total processing time spent on that specific scene.

In [ ]:
import pandas as pd

results_df = pd.read_csv(METRICS_CSV)
display(results_df)